In [0]:
spark

In [0]:
spark.conf.set(
  "fs.azure.account.key.sthealthcarecap.dfs.core.windows.net",
  "AhACx90sznulBXEHB/jIcBocuKHVxA9hponltHxpuXromyB2TAL5GicTI+xTP52RRsUSvqYcyb+/+ASt3pfr2w=="
)

In [0]:
from pyspark.sql.functions import *

In [0]:
patients_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/raw/patients/patient_master_seed.csv")

display(patients_df)

In [0]:
doctors_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/raw/doctors/doctor_master_seed.csv")

display(doctors_df)

In [0]:
visits_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/raw/visits/visit_transactions.csv")

display(visits_df)

In [0]:
claims_df = spark.read.format("csv") \
.option("header","true") \
.option("inferSchema","true") \
.load("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/raw/claims/insurance_claims.csv")

display(claims_df)

In [0]:
patients_bronze = patients_df \
.withColumn("ingestion_time", current_timestamp()) \
.withColumn("source_file", lit("patient_master_seed.csv"))

doctors_bronze = doctors_df \
.withColumn("ingestion_time", current_timestamp()) \
.withColumn("source_file", lit("doctor_master_seed.csv"))

visits_bronze = visits_df \
.withColumn("ingestion_time", current_timestamp()) \
.withColumn("source_file", lit("visit_transactions.csv"))

claims_bronze = claims_df \
.withColumn("ingestion_time", current_timestamp()) \
.withColumn("source_file", lit("insurance_claims.csv"))

In [0]:
patients_bronze.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_patients")

doctors_bronze.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_doctors")

visits_bronze.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_visits")

claims_bronze.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_claims")

In [0]:
display(
    dbutils.fs.ls(
        "abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/"
    )
)

In [0]:
from pyspark.sql.functions import *

In [0]:
patients_df = spark.read.format("delta") \
.load("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_patients")

doctors_df = spark.read.format("delta") \
.load("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_doctors")

visits_df = spark.read.format("delta") \
.load("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_visits")

claims_df = spark.read.format("delta") \
.load("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_claims")

In [0]:
patients_clean = patients_df.dropDuplicates(["patient_id"])

doctors_clean = doctors_df.dropDuplicates(["doctor_id"])

visits_clean = visits_df.dropDuplicates(["visit_id"])

claims_clean = claims_df.dropDuplicates(["claim_id"])

In [0]:
valid_visits = visits_clean.filter(
    col("patient_id").isNotNull() &
    col("doctor_id").isNotNull() &
    col("billing_amount").isNotNull()
)

invalid_visits = visits_clean.filter(
    col("patient_id").isNull() |
    col("doctor_id").isNull() |
    col("billing_amount").isNull()
)

In [0]:
invalid_billing = visits_clean.filter(
    col("billing_amount") <= 0
)

In [0]:
valid_claims = claims_clean.filter(
    col("claim_amount") > 0
)

invalid_claims = claims_clean.filter(
    col("claim_amount") <= 0
)

In [0]:
patient_visit_join = valid_visits.join(
    patients_clean.drop("ingestion_time", "source_file"),
    "patient_id",
    "left"
)

doctor_visit_join = patient_visit_join.join(
    doctors_clean.drop("ingestion_time", "source_file", "department", "last_updated"),
    "doctor_id",
    "left"
)

final_silver = doctor_visit_join.join(
    valid_claims.drop("ingestion_time", "source_file", "patient_id", "department", "insurance_plan", "last_updated"),
    "visit_id",
    "left"
)

display(final_silver)

In [0]:
patients_clean.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/silver/silver_patients")

doctors_clean.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/silver/silver_doctors")

final_silver.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/silver/silver_visits_claims")

In [0]:
invalid_visits \
.withColumn("rejection_reason", lit("Null mandatory fields")) \
.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/rejected/rejected_visits")

In [0]:
invalid_billing \
.withColumn("rejection_reason", lit("Invalid billing amount")) \
.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/rejected/rejected_billing")

In [0]:
invalid_claims \
.withColumn("rejection_reason", lit("Invalid claim amount")) \
.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/rejected/rejected_claims")

In [0]:
display(
    dbutils.fs.ls(
        "abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/rejected/"
    )
)

In [0]:
silver_df = spark.read.format("delta") \
.load("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/silver/silver_visits_claims")

In [0]:
department_revenue = silver_df.groupBy("department") \
.agg(
    sum("billing_amount").alias("total_revenue")
)

display(department_revenue)

In [0]:
doctor_performance = silver_df.groupBy(
    "doctor_id",
    "doctor_name",
    "department"
).agg(
    count("visit_id").alias("visit_count"),
    sum("billing_amount").alias("total_billing")
)

display(doctor_performance)

In [0]:
insurance_summary = silver_df.groupBy("insurance_plan") \
.agg(
    count("claim_id").alias("total_claims"),
    sum("claim_amount").alias("total_claim_amount")
)

display(insurance_summary)

In [0]:
insurance_summary = silver_df.groupBy("insurance_plan") \
.agg(
    count("claim_id").alias("total_claims"),
    sum("claim_amount").alias("total_claim_amount")
)

display(insurance_summary)

In [0]:
department_revenue.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/gold/gold_department_revenue")

doctor_performance.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/gold/gold_doctor_performance_summary")

insurance_summary.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/gold/gold_insurance_claim_summary")

In [0]:
silver_patients = spark.read.format("delta") \
.load("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/silver/silver_patients")

In [0]:
scd2_patients = silver_patients \
.withColumn("effective_date", current_date()) \
.withColumn("expiry_date", lit(None).cast("date")) \
.withColumn("is_current", lit(True))

In [0]:
scd2_patients.write.format("delta") \
.mode("overwrite") \
.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/gold/dim_patients_scd2")

In [0]:
scd2_patients.createOrReplaceTempView("dim_patients_scd2")

In [0]:
%sql

MERGE INTO delta.`abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/gold/dim_patients_scd2` target
USING dim_patients_scd2 source
ON target.patient_id = source.patient_id
WHEN MATCHED THEN
UPDATE SET
target.city = source.city
WHEN NOT MATCHED THEN
INSERT *

In [0]:
%sql

DESCRIBE HISTORY delta.`abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/gold/dim_patients_scd2`

In [0]:
%sql

SELECT *
FROM delta.`abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/gold/dim_patients_scd2`
VERSION AS OF 0

In [0]:
%sql

SELECT *
FROM delta.`abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/gold/dim_patients_scd2`
VERSION AS OF 0

In [0]:
%sql

OPTIMIZE delta.`abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/gold/dim_patients_scd2`

In [0]:
%sql

VACUUM delta.`abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/gold/dim_patients_scd2`
RETAIN 168 HOURS

In [0]:
#doctor_performance.write.format("parquet") \
#.mode("overwrite") \
#.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/reporting/doctor_performance")
from pyspark.sql.functions import *
import os

temp_path = "abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/reporting/temp_doctor_performance"

final_path = "abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/reporting/doctor_performance/doctor_performance.parquet"

doctor_performance.coalesce(1) \
.write.mode("overwrite") \
.format("parquet") \
.save(temp_path)

files = dbutils.fs.ls(temp_path)

parquet_file = [file.path for file in files if file.path.endswith(".parquet")][0]

dbutils.fs.cp(parquet_file, final_path)

dbutils.fs.rm(temp_path, recurse=True)

print("doctor_performance.parquet created successfully")

In [0]:
#department_revenue.write.format("parquet") \
#.mode("overwrite") \
#.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/reporting/department_revenue")
from pyspark.sql.functions import *
import os

temp_path = "abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/reporting/temp_department_revenue"

final_path = "abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/reporting/department_revenue/department_revenue.parquet"

department_revenue.coalesce(1) \
.write.mode("overwrite") \
.format("parquet") \
.save(temp_path)

files = dbutils.fs.ls(temp_path)

parquet_file = [file.path for file in files if file.path.endswith(".parquet")][0]

dbutils.fs.cp(parquet_file, final_path)

dbutils.fs.rm(temp_path, recurse=True)

print("department_revenue.parquet created successfully")

In [0]:
#insurance_summary.write.format("parquet") \
#.mode("overwrite") \
#.save("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/reporting/insurance_summary")

from pyspark.sql.functions import *
import os

temp_path = "abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/reporting/temp_insurance_summary"

final_path = "abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/reporting/insurance_summary/insurance_summary.parquet"

insurance_summary.coalesce(1) \
.write.mode("overwrite") \
.format("parquet") \
.save(temp_path)

files = dbutils.fs.ls(temp_path)

parquet_file = [file.path for file in files if file.path.endswith(".parquet")][0]

dbutils.fs.cp(parquet_file, final_path)

dbutils.fs.rm(temp_path, recurse=True)

print("insurance_summary.parquet created successfully")

In [0]:
display(
    dbutils.fs.ls(
        "abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/reporting/"
    )
)

In [0]:
display(
    dbutils.fs.ls(
        "abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/"
    )
)

path,name,size,modificationTime
abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_claims/,bronze_claims/,0,1779356228000
abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_doctors/,bronze_doctors/,0,1779356224000
abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_patients/,bronze_patients/,0,1779356219000
abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_visits/,bronze_visits/,0,1779356227000


In [0]:
bronze_visits = spark.read.format("delta") \
.load("abfss://healthcare-data@sthealthcarecap.dfs.core.windows.net/bronze/bronze_visits")

display(bronze_visits)

visit_id,visit_date,patient_id,doctor_id,department,visit_type,diagnosis_code,billing_amount,payment_status,ingestion_date,ingestion_time,source_file
VIS000001,2025-09-16,PAT00173,DOC0050,Oncology,Emergency,DX269,26021.88,Pending,2026-02-22,2026-05-21T10:37:26.670941Z,visit_transactions.csv
VIS000002,2025-11-26,PAT00037,DOC0150,Oncology,OPD,DX816,84454.95,Paid,2026-02-16,2026-05-21T10:37:26.670941Z,visit_transactions.csv
VIS000003,2025-09-04,PAT00396,DOC0141,General Medicine,OPD,DX340,67757.96,Pending,2026-03-10,2026-05-21T10:37:26.670941Z,visit_transactions.csv
VIS000004,2026-03-30,PAT00132,DOC0050,Oncology,Emergency,DX895,40851.15,Rejected,2026-02-07,2026-05-21T10:37:26.670941Z,visit_transactions.csv
VIS000005,2025-12-29,PAT00847,DOC0035,General Medicine,Diagnostic,DX291,17864.7,Rejected,2026-01-15,2026-05-21T10:37:26.670941Z,visit_transactions.csv
VIS000006,2026-01-05,PAT00019,DOC0099,Dermatology,Follow-up,DX292,38455.57,Rejected,2026-02-13,2026-05-21T10:37:26.670941Z,visit_transactions.csv
VIS000007,2025-11-21,PAT00901,DOC0068,Pediatrics,Diagnostic,DX293,12481.21,Insurance Submitted,2026-04-19,2026-05-21T10:37:26.670941Z,visit_transactions.csv
VIS000008,2025-11-30,PAT00162,DOC0169,General Medicine,Follow-up,DX920,56160.65,Pending,2026-02-12,2026-05-21T10:37:26.670941Z,visit_transactions.csv
VIS000009,2025-12-14,PAT00451,DOC0123,Cardiology,Diagnostic,DX976,30996.43,Rejected,2026-02-01,2026-05-21T10:37:26.670941Z,visit_transactions.csv
VIS000010,2025-10-26,PAT00290,DOC0060,Neurology,Emergency,DX790,61995.46,Pending,2026-03-09,2026-05-21T10:37:26.670941Z,visit_transactions.csv


In [0]:
print("Visits Count:", bronze_visits.count())

Visits Count: 2500
